# MovieLens 20M: Recommendation Systems and Clustering Analysis

**Self-Contained Notebook**

All source code is embedded in this notebook for complete reproducibility.

## Algorithms Implemented:

**Recommendation Systems:**
- SVD (Traditional Matrix Factorization)
- PageRank (Graph-based)
- Hybrid (SVD + PageRank)
- ALS (Modern Matrix Factorization)
- ItemKNN (Traditional Collaborative Filtering)

**Clustering:**
- K-means, Hierarchical, PCA

## Evaluation Metrics:

**Rating:** MAE, RMSE
**Ranking:** Precision@K, Recall@K, NDCG@K, HitRate@K
**Clustering:** Silhouette, Davies-Bouldin, Calinski-Harabasz

## Module 1: Data Loader

In [ ]:
import os
import pandas as pd
import numpy as np
from typing import Tuple, Optional


class MovieLensLoader:

    def __init__(self, data_dir: str = './data'):
        self.data_dir = data_dir
        self.ratings = None
        self.movies = None
        self.tags = None
        self.genome_scores = None

    def load_data(self, sample_size: Optional[int] = None) -> None:
        print("Loading data...")

        ratings_path = os.path.join(self.data_dir, 'ml-20m', 'ratings.csv')
        if sample_size:
            self.ratings = pd.read_csv(ratings_path, nrows=sample_size)
        else:
            self.ratings = pd.read_csv(ratings_path)
        print(f"Loaded {len(self.ratings)} ratings")

        movies_path = os.path.join(self.data_dir, 'ml-20m', 'movies.csv')
        self.movies = pd.read_csv(movies_path)
        print(f"Loaded {len(self.movies)} ")

        try:
            tags_path = os.path.join(self.data_dir, 'ml-20m', 'tags.csv')
            self.tags = pd.read_csv(tags_path)
            print(f"Loaded {len(self.tags)} ")
        except FileNotFoundError:
            print("，")

        try:
            genome_path = os.path.join(self.data_dir, 'ml-20m', 'genome-scores.csv')
            if sample_size:
                self.genome_scores = pd.read_csv(genome_path, nrows=sample_size*100)
            else:
                self.genome_scores = pd.read_csv(genome_path)
            print(f"Loaded {len(self.genome_scores)}  genome scores")
        except FileNotFoundError:
            print("Genome scores ，")

    def preprocess_for_recommendation(self,
                                     min_user_ratings: int = 20,
                                     min_movie_ratings: int = 20) -> Tuple[pd.DataFrame, np.ndarray]:
        print("\nPreprocessing data...")

        user_counts = self.ratings['userId'].value_counts()
        movie_counts = self.ratings['movieId'].value_counts()

        active_users = user_counts[user_counts >= min_user_ratings].index
        popular_movies = movie_counts[movie_counts >= min_movie_ratings].index

        filtered_ratings = self.ratings[
            (self.ratings['userId'].isin(active_users)) &
            (self.ratings['movieId'].isin(popular_movies))
        ]

        print(f": {len(filtered_ratings)} , "
              f"{filtered_ratings['userId'].nunique()} , "
              f"{filtered_ratings['movieId'].nunique()} ")

        user_movie_matrix = filtered_ratings.pivot_table(
            index='userId',
            columns='movieId',
            values='rating',
            fill_value=0
        )

        return user_movie_matrix, filtered_ratings

    def create_movie_features(self, use_genome: bool = True) -> pd.DataFrame:
        print("\n...")

        if use_genome and self.genome_scores is not None:
            genome_pivot = self.genome_scores.pivot_table(
                index='movieId',
                columns='tagId',
                values='relevance',
                fill_value=0
            )

            common_movies = list(set(genome_pivot.index) & set(self.movies['movieId']))
            movie_features = genome_pivot.loc[common_movies]

            print(f" {movie_features.shape[0]}  {movie_features.shape[1]} ")

        else:
            print("...")

            all_genres = set()
            for genres in self.movies['genres']:
                if genres != '(no genres listed)':
                    all_genres.update(genres.split('|'))
            all_genres = sorted(list(all_genres))

            genre_features = []
            movie_ids = []

            for _, row in self.movies.iterrows():
                genres = row['genres'].split('|')
                feature = [1 if g in genres else 0 for g in all_genres]
                genre_features.append(feature)
                movie_ids.append(row['movieId'])

            movie_features = pd.DataFrame(
                genre_features,
                index=movie_ids,
                columns=all_genres
            )

            print(f" {movie_features.shape[0]}  {movie_features.shape[1]} （）")

        return movie_features

    def get_movie_info(self, movie_ids):
        return self.movies[self.movies['movieId'].isin(movie_ids)]


## Module 2: Evaluation Metrics

In [ ]:
import numpy as np
import pandas as pd
from typing import Dict, List, Tuple, Set
from collections import defaultdict


class RecommenderEvaluator:

    def __init__(self, k: int = 10):
        self.k = k

    @staticmethod
    def rating_metrics(predictions: np.ndarray, actuals: np.ndarray) -> Dict[str, float]:
        mae = np.mean(np.abs(predictions - actuals))
        rmse = np.sqrt(np.mean((predictions - actuals) ** 2))

        return {
            'MAE': mae,
            'RMSE': rmse,
            'n_samples': len(predictions)
        }

    @staticmethod
    def precision_at_k(recommended: List, relevant: Set, k: int = 10) -> float:
        recommended_k = recommended[:k]
        relevant_count = len([item for item in recommended_k if item in relevant])
        return relevant_count / k if k > 0 else 0.0

    @staticmethod
    def recall_at_k(recommended: List, relevant: Set, k: int = 10) -> float:
        if len(relevant) == 0:
            return 0.0

        recommended_k = recommended[:k]
        relevant_count = len([item for item in recommended_k if item in relevant])
        return relevant_count / len(relevant)

    @staticmethod
    def f1_at_k(recommended: List, relevant: Set, k: int = 10) -> float:
        precision = RecommenderEvaluator.precision_at_k(recommended, relevant, k)
        recall = RecommenderEvaluator.recall_at_k(recommended, relevant, k)

        if precision + recall == 0:
            return 0.0

        return 2 * precision * recall / (precision + recall)

    @staticmethod
    def ndcg_at_k(recommended: List, relevant: Set, k: int = 10) -> float:
        recommended_k = recommended[:k]

        dcg = 0.0
        for i, item in enumerate(recommended_k):
            if item in relevant:
                dcg += 1.0 / np.log2(i + 2)

        idcg = 0.0
        for i in range(min(len(relevant), k)):
            idcg += 1.0 / np.log2(i + 2)

        if idcg == 0:
            return 0.0

        return dcg / idcg

    @staticmethod
    def hit_rate_at_k(recommended: List, relevant: Set, k: int = 10) -> float:
        recommended_k = recommended[:k]
        for item in recommended_k:
            if item in relevant:
                return 1.0
        return 0.0

    @staticmethod
    def mrr(recommended: List, relevant: Set) -> float:
        for i, item in enumerate(recommended):
            if item in relevant:
                return 1.0 / (i + 1)
        return 0.0

    @staticmethod
    def average_precision(recommended: List, relevant: Set) -> float:
        if len(relevant) == 0:
            return 0.0

        precision_sum = 0.0
        relevant_count = 0

        for i, item in enumerate(recommended):
            if item in relevant:
                relevant_count += 1
                precision_at_i = relevant_count / (i + 1)
                precision_sum += precision_at_i

        return precision_sum / len(relevant)

    def evaluate_recommendations(self,
                                 user_recommendations: Dict[int, List[int]],
                                 user_relevant_items: Dict[int, Set[int]],
                                 k: int = None) -> Dict[str, float]:
        if k is None:
            k = self.k

        precisions = []
        recalls = []
        f1_scores = []
        ndcgs = []
        hit_rates = []
        mrrs = []
        aps = []

        for user_id in user_recommendations:
            if user_id not in user_relevant_items:
                continue

            recommended = user_recommendations[user_id]
            relevant = user_relevant_items[user_id]

            if len(relevant) == 0:
                continue

            precisions.append(self.precision_at_k(recommended, relevant, k))
            recalls.append(self.recall_at_k(recommended, relevant, k))
            f1_scores.append(self.f1_at_k(recommended, relevant, k))
            ndcgs.append(self.ndcg_at_k(recommended, relevant, k))
            hit_rates.append(self.hit_rate_at_k(recommended, relevant, k))
            mrrs.append(self.mrr(recommended, relevant))
            aps.append(self.average_precision(recommended, relevant))

        metrics = {
            f'Precision@{k}': np.mean(precisions) if precisions else 0.0,
            f'Recall@{k}': np.mean(recalls) if recalls else 0.0,
            f'F1@{k}': np.mean(f1_scores) if f1_scores else 0.0,
            f'NDCG@{k}': np.mean(ndcgs) if ndcgs else 0.0,
            f'HitRate@{k}': np.mean(hit_rates) if hit_rates else 0.0,
            'MRR': np.mean(mrrs) if mrrs else 0.0,
            'MAP': np.mean(aps) if aps else 0.0,
            'n_users': len(precisions)
        }

        return metrics

    @staticmethod
    def catalog_coverage(all_recommendations: List[int],
                        catalog_size: int) -> float:
        unique_items = set(all_recommendations)
        return len(unique_items) / catalog_size if catalog_size > 0 else 0.0

    @staticmethod
    def diversity(recommendations: List[List[int]],
                  similarity_matrix: np.ndarray = None) -> float:
        if similarity_matrix is not None:
            total_dissimilarity = 0.0
            count = 0

            for rec_list in recommendations:
                for i in range(len(rec_list)):
                    for j in range(i + 1, len(rec_list)):
                        if i < len(similarity_matrix) and j < len(similarity_matrix):
                            dissimilarity = 1 - similarity_matrix[rec_list[i], rec_list[j]]
                            total_dissimilarity += dissimilarity
                            count += 1

            return total_dissimilarity / count if count > 0 else 0.0
        else:
            all_items = []
            for rec_list in recommendations:
                all_items.extend(rec_list)

            if len(all_items) == 0:
                return 0.0

            return len(set(all_items)) / len(all_items)


def create_relevance_set(test_ratings: pd.DataFrame,
                        threshold: float = 4.0) -> Dict[int, Set[int]]:
    relevance = defaultdict(set)

    for _, row in test_ratings.iterrows():
        if row['rating'] >= threshold:
            relevance[row['userId']].add(row['movieId'])

    return dict(relevance)


## Module 3: Recommender Systems

In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
from typing import List, Tuple, Dict
import time
import networkx as nx
from tqdm import tqdm


class SVDRecommender:

    def __init__(self, n_components: int = 50):
        self.n_components = n_components
        self.svd = TruncatedSVD(n_components=n_components, random_state=42)
        self.user_factors = None
        self.movie_factors = None
        self.user_movie_matrix = None
        self.user_ids = None
        self.movie_ids = None

    def fit(self, user_movie_matrix: pd.DataFrame) -> None:
        print(f"\nTraining SVD recommender (n_components={self.n_components})...")
        start_time = time.time()

        self.user_movie_matrix = user_movie_matrix
        self.user_ids = user_movie_matrix.index
        self.movie_ids = user_movie_matrix.columns

        # U * Sigma * V^T ≈ R
        self.user_factors = self.svd.fit_transform(user_movie_matrix)

        self.movie_factors = self.svd.components_.T

        explained_variance = np.sum(self.svd.explained_variance_ratio_)
        print(f"Training complete！ {time.time() - start_time:.2f} ")
        print(f": {explained_variance:.4f}")

    def predict_rating(self, user_idx: int, movie_idx: int) -> float:
        prediction = np.dot(self.user_factors[user_idx], self.movie_factors[movie_idx])
        return np.clip(prediction, 0.5, 5.0)

    def recommend_for_user(self, user_id: int, top_n: int = 10,
                          exclude_rated: bool = True) -> List[Tuple[int, float]]:
        if user_id not in self.user_ids:
            raise ValueError(f" {user_id} ")

        user_idx = self.user_ids.get_loc(user_id)

        predictions = np.dot(self.user_factors[user_idx], self.movie_factors.T)
        predictions = np.clip(predictions, 0.5, 5.0)

        if exclude_rated:
            rated_mask = self.user_movie_matrix.iloc[user_idx] > 0
            predictions[rated_mask] = -1

        top_indices = np.argsort(predictions)[::-1][:top_n]
        recommendations = [
            (self.movie_ids[idx], predictions[idx])
            for idx in top_indices
            if predictions[idx] > 0
        ]

        return recommendations

    def find_similar_movies(self, movie_id: int, top_n: int = 10) -> List[Tuple[int, float]]:
        if movie_id not in self.movie_ids:
            raise ValueError(f" {movie_id} ")

        movie_idx = self.movie_ids.get_loc(movie_id)

        movie_vector = self.movie_factors[movie_idx].reshape(1, -1)
        similarities = cosine_similarity(movie_vector, self.movie_factors)[0]

        similar_indices = np.argsort(similarities)[::-1][1:top_n+1]
        similar_movies = [
            (self.movie_ids[idx], similarities[idx])
            for idx in similar_indices
        ]

        return similar_movies

    def evaluate(self, test_ratings: pd.DataFrame) -> Dict[str, float]:
        print("\n...")

        predictions = []
        actuals = []

        for _, row in test_ratings.iterrows():
            user_id = row['userId']
            movie_id = row['movieId']
            actual_rating = row['rating']

            try:
                user_idx = self.user_ids.get_loc(user_id)
                movie_idx = self.movie_ids.get_loc(movie_id)
                pred_rating = self.predict_rating(user_idx, movie_idx)

                predictions.append(pred_rating)
                actuals.append(actual_rating)
            except KeyError:
                continue

        predictions = np.array(predictions)
        actuals = np.array(actuals)

        mae = np.mean(np.abs(predictions - actuals))
        rmse = np.sqrt(np.mean((predictions - actuals) ** 2))

        metrics = {
            'MAE': mae,
            'RMSE': rmse,
            'n_predictions': len(predictions)
        }

        print(f"MAE: {mae:.4f}")
        print(f"RMSE: {rmse:.4f}")
        print(f": {len(predictions)}")

        return metrics


class CollaborativeFiltering:

    def __init__(self, similarity_type: str = 'cosine'):
        self.similarity_type = similarity_type
        self.user_similarity = None
        self.movie_similarity = None
        self.user_movie_matrix = None

    def fit(self, user_movie_matrix: pd.DataFrame) -> None:
        print(f"\n{self.similarity_type}...")
        start_time = time.time()

        self.user_movie_matrix = user_movie_matrix

        self.user_similarity = cosine_similarity(user_movie_matrix)

        self.movie_similarity = cosine_similarity(user_movie_matrix.T)

        print(f"！ {time.time() - start_time:.2f} ")

    def recommend_user_based(self, user_id: int, top_n: int = 10) -> List[Tuple[int, float]]:
        if user_id not in self.user_movie_matrix.index:
            raise ValueError(f" {user_id} ")

        user_idx = self.user_movie_matrix.index.get_loc(user_id)

        similar_users = self.user_similarity[user_idx]

        predictions = np.dot(similar_users, self.user_movie_matrix.values)
        predictions = predictions / (np.sum(np.abs(similar_users)) + 1e-8)

        rated_mask = self.user_movie_matrix.iloc[user_idx] > 0
        predictions[rated_mask] = -1

        top_indices = np.argsort(predictions)[::-1][:top_n]
        recommendations = [
            (self.user_movie_matrix.columns[idx], predictions[idx])
            for idx in top_indices
            if predictions[idx] > 0
        ]

        return recommendations


class PageRankRecommender:

    def __init__(self, alpha: float = 0.85, cf_weight: float = 0.5):
        self.alpha = alpha
        self.cf_weight = cf_weight
        self.user_movie_matrix = None
        self.graph = None
        self.pagerank_scores = None
        self.cosine_sim_matrix = None
        self.user_ids = None
        self.movie_ids = None

    def fit(self, user_movie_matrix: pd.DataFrame) -> None:
        print(f"\nTraining PageRank recommender (alpha={self.alpha}, cf_weight={self.cf_weight})...")
        start_time = time.time()

        self.user_movie_matrix = user_movie_matrix
        self.user_ids = user_movie_matrix.index
        self.movie_ids = user_movie_matrix.columns

        print("  - ...")
        normalized_data = normalize(user_movie_matrix, axis=0)
        self.cosine_sim_matrix = cosine_similarity(normalized_data.T)
        self.cosine_sim_df = pd.DataFrame(
            self.cosine_sim_matrix,
            index=self.movie_ids,
            columns=self.movie_ids
        )

        print("  - -...")
        self.graph = nx.Graph()

        self.graph.add_nodes_from(self.user_ids, bipartite=0)
        self.graph.add_nodes_from(self.movie_ids, bipartite=1)

        edge_count = 0
        for user in tqdm(self.user_ids, desc="  - "):
            for movie in self.movie_ids:
                rating = user_movie_matrix.loc[user, movie]
                if rating > 0:  # 
                    self.graph.add_edge(user, movie, weight=rating)
                    edge_count += 1

        print(f"     {len(self.graph.nodes)} , {edge_count} ")

        print("  -  PageRank ...")
        pagerank_all = nx.pagerank(self.graph, alpha=self.alpha)

        self.pagerank_scores = {
            movie: score
            for movie, score in pagerank_all.items()
            if movie in self.movie_ids
        }

        print(f"Training complete！ {time.time() - start_time:.2f} ")

    def recommend_for_user(self, user_id: int, top_n: int = 10,
                          exclude_rated: bool = True) -> List[Tuple[int, float]]:
        if user_id not in self.user_ids:
            raise ValueError(f" {user_id} ")

        user_idx = self.user_ids.get_loc(user_id)
        user_ratings = self.user_movie_matrix.loc[user_id]

        rated_movies = user_ratings[user_ratings > 0].index.tolist()

        cf_scores = {}
        for movie in self.movie_ids:
            if exclude_rated and movie in rated_movies:
                continue

            similarity_scores = []
            for rated_movie in rated_movies:
                sim = self.cosine_sim_df.loc[movie, rated_movie]
                rating = user_ratings[rated_movie]
                similarity_scores.append(sim * rating)

            if similarity_scores:
                cf_scores[movie] = np.mean(similarity_scores)
            else:
                cf_scores[movie] = 0

        if cf_scores:
            max_cf = max(cf_scores.values()) if cf_scores.values() else 1
            if max_cf > 0:
                cf_scores = {k: v / max_cf for k, v in cf_scores.items()}

        max_pr = max(self.pagerank_scores.values()) if self.pagerank_scores else 1
        normalized_pr_scores = {
            k: v / max_pr for k, v in self.pagerank_scores.items()
        }

        combined_scores = {}
        for movie in self.movie_ids:
            if exclude_rated and movie in rated_movies:
                continue

            cf_score = cf_scores.get(movie, 0)
            pr_score = normalized_pr_scores.get(movie, 0)

            combined_score = (
                self.cf_weight * cf_score +
                (1 - self.cf_weight) * pr_score
            )
            combined_scores[movie] = combined_score

        top_movies = sorted(
            combined_scores.items(),
            key=lambda x: x[1],
            reverse=True
        )[:top_n]

        return top_movies

    def get_movie_pagerank(self, movie_id: int) -> float:
        if movie_id not in self.movie_ids:
            raise ValueError(f" {movie_id} ")
        return self.pagerank_scores.get(movie_id, 0)

    def evaluate(self, test_ratings: pd.DataFrame) -> Dict[str, float]:
        print("\n PageRank ...")

        predictions = []
        actuals = []

        for _, row in test_ratings.iterrows():
            user_id = row['userId']
            movie_id = row['movieId']
            actual_rating = row['rating']

            try:
                user_idx = self.user_ids.get_loc(user_id)
                user_ratings = self.user_movie_matrix.loc[user_id]
                rated_movies = user_ratings[user_ratings > 0].index.tolist()

                similarity_scores = []
                for rated_movie in rated_movies:
                    if rated_movie != movie_id and movie_id in self.movie_ids:
                        sim = self.cosine_sim_df.loc[movie_id, rated_movie]
                        rating = user_ratings[rated_movie]
                        similarity_scores.append(sim * rating)

                if similarity_scores:
                    pred_rating = np.mean(similarity_scores)
                    pred_rating = np.clip(pred_rating, 0.5, 5.0)

                    predictions.append(pred_rating)
                    actuals.append(actual_rating)

            except (KeyError, ValueError):
                continue

        if len(predictions) > 0:
            predictions = np.array(predictions)
            actuals = np.array(actuals)

            mae = np.mean(np.abs(predictions - actuals))
            rmse = np.sqrt(np.mean((predictions - actuals) ** 2))

            metrics = {
                'MAE': mae,
                'RMSE': rmse,
                'n_predictions': len(predictions)
            }

            print(f"MAE: {mae:.4f}")
            print(f"RMSE: {rmse:.4f}")
            print(f": {len(predictions)}")

            return metrics
        else:
            print("：")
            return {'MAE': float('inf'), 'RMSE': float('inf'), 'n_predictions': 0}


class HybridRecommender:

    def __init__(self, svd_recommender: SVDRecommender,
                 pagerank_recommender: PageRankRecommender,
                 svd_weight: float = 0.5):
        self.svd_recommender = svd_recommender
        self.pagerank_recommender = pagerank_recommender
        self.svd_weight = svd_weight

    def recommend_for_user(self, user_id: int, top_n: int = 10,
                          exclude_rated: bool = True) -> List[Tuple[int, float]]:
        svd_recs = self.svd_recommender.recommend_for_user(
            user_id, top_n=top_n*2, exclude_rated=exclude_rated
        )
        svd_scores = dict(svd_recs)

        pr_recs = self.pagerank_recommender.recommend_for_user(
            user_id, top_n=top_n*2, exclude_rated=exclude_rated
        )
        pr_scores = dict(pr_recs)

        all_movies = set(svd_scores.keys()) | set(pr_scores.keys())

        max_svd = max(svd_scores.values()) if svd_scores else 1
        max_pr = max(pr_scores.values()) if pr_scores else 1

        combined_scores = {}
        for movie in all_movies:
            svd_score = svd_scores.get(movie, 0) / max_svd
            pr_score = pr_scores.get(movie, 0) / max_pr

            combined_score = (
                self.svd_weight * svd_score +
                (1 - self.svd_weight) * pr_score
            )
            combined_scores[movie] = combined_score

        top_movies = sorted(
            combined_scores.items(),
            key=lambda x: x[1],
            reverse=True
        )[:top_n]

        return top_movies

    def evaluate(self, test_ratings: pd.DataFrame) -> Dict[str, float]:
        print(f"\n (SVD={self.svd_weight})...")

        predictions = []
        actuals = []

        for _, row in test_ratings.iterrows():
            user_id = row['userId']
            movie_id = row['movieId']
            actual_rating = row['rating']

            try:
                user_idx = self.svd_recommender.user_ids.get_loc(user_id)
                movie_idx = self.svd_recommender.movie_ids.get_loc(movie_id)
                svd_pred = self.svd_recommender.predict_rating(user_idx, movie_idx)

                pr_user_idx = self.pagerank_recommender.user_ids.get_loc(user_id)
                user_ratings = self.pagerank_recommender.user_movie_matrix.loc[user_id]
                rated_movies = user_ratings[user_ratings > 0].index.tolist()

                similarity_scores = []
                for rated_movie in rated_movies:
                    if rated_movie != movie_id:
                        sim = self.pagerank_recommender.cosine_sim_df.loc[movie_id, rated_movie]
                        rating = user_ratings[rated_movie]
                        similarity_scores.append(sim * rating)

                if similarity_scores:
                    pr_pred = np.mean(similarity_scores)
                    pr_pred = np.clip(pr_pred, 0.5, 5.0)
                else:
                    pr_pred = 3.0  # 

                hybrid_pred = self.svd_weight * svd_pred + (1 - self.svd_weight) * pr_pred
                hybrid_pred = np.clip(hybrid_pred, 0.5, 5.0)

                predictions.append(hybrid_pred)
                actuals.append(actual_rating)

            except (KeyError, ValueError):
                continue

        if len(predictions) > 0:
            predictions = np.array(predictions)
            actuals = np.array(actuals)

            mae = np.mean(np.abs(predictions - actuals))
            rmse = np.sqrt(np.mean((predictions - actuals) ** 2))

            metrics = {
                'MAE': mae,
                'RMSE': rmse,
                'n_predictions': len(predictions)
            }

            print(f"MAE: {mae:.4f}")
            print(f"RMSE: {rmse:.4f}")
            print(f": {len(predictions)}")

            return metrics
        else:
            print("：")
            return {'MAE': float('inf'), 'RMSE': float('inf'), 'n_predictions': 0}

class ALSRecommender:

    def __init__(self, n_factors: int = 50, n_iterations: int = 15, regularization: float = 0.01):
        self.n_factors = n_factors
        self.n_iterations = n_iterations
        self.regularization = regularization
        self.user_factors = None
        self.item_factors = None
        self.user_ids = None
        self.movie_ids = None
        self.user_movie_matrix = None

    def fit(self, user_movie_matrix: pd.DataFrame) -> None:
        print(f"\nTraining ALS recommender (factors={self.n_factors}, iterations={self.n_iterations})...")
        start_time = time.time()

        self.user_movie_matrix = user_movie_matrix
        self.user_ids = user_movie_matrix.index
        self.movie_ids = user_movie_matrix.columns

        R = user_movie_matrix.values
        n_users, n_items = R.shape

        np.random.seed(42)
        self.user_factors = np.random.normal(0, 0.1, (n_users, self.n_factors))
        self.item_factors = np.random.normal(0, 0.1, (n_items, self.n_factors))

        for iteration in range(self.n_iterations):
            for u in range(n_users):
                rated_items = R[u, :] > 0
                if not np.any(rated_items):
                    continue

                item_factors_u = self.item_factors[rated_items, :]
                ratings_u = R[u, rated_items]

                A = item_factors_u.T @ item_factors_u + self.regularization * np.eye(self.n_factors)
                b = item_factors_u.T @ ratings_u
                self.user_factors[u, :] = np.linalg.solve(A, b)

            for i in range(n_items):
                rating_users = R[:, i] > 0
                if not np.any(rating_users):
                    continue

                user_factors_i = self.user_factors[rating_users, :]
                ratings_i = R[rating_users, i]

                A = user_factors_i.T @ user_factors_i + self.regularization * np.eye(self.n_factors)
                b = user_factors_i.T @ ratings_i
                self.item_factors[i, :] = np.linalg.solve(A, b)

        print(f"Training complete！ {time.time() - start_time:.2f} ")

    def predict_rating(self, user_idx: int, movie_idx: int) -> float:
        prediction = np.dot(self.user_factors[user_idx], self.item_factors[movie_idx])
        return np.clip(prediction, 0.5, 5.0)

    def recommend_for_user(self, user_id: int, top_n: int = 10,
                          exclude_rated: bool = True) -> List[Tuple[int, float]]:
        if user_id not in self.user_ids:
            raise ValueError(f" {user_id} ")

        user_idx = self.user_ids.get_loc(user_id)

        predictions = np.dot(self.user_factors[user_idx], self.item_factors.T)
        predictions = np.clip(predictions, 0.5, 5.0)

        if exclude_rated:
            rated_mask = self.user_movie_matrix.iloc[user_idx] > 0
            predictions[rated_mask] = -1

        top_indices = np.argsort(predictions)[::-1][:top_n]
        recommendations = [
            (self.movie_ids[idx], predictions[idx])
            for idx in top_indices
        ]

        return recommendations

    def evaluate(self, test_ratings: pd.DataFrame) -> Dict[str, float]:
        print("\n ALS ...")

        predictions = []
        actuals = []

        for _, row in test_ratings.iterrows():
            user_id = row['userId']
            movie_id = row['movieId']
            actual_rating = row['rating']

            try:
                user_idx = self.user_ids.get_loc(user_id)
                movie_idx = self.movie_ids.get_loc(movie_id)
                pred_rating = self.predict_rating(user_idx, movie_idx)

                predictions.append(pred_rating)
                actuals.append(actual_rating)
            except KeyError:
                continue

        predictions = np.array(predictions)
        actuals = np.array(actuals)

        mae = np.mean(np.abs(predictions - actuals))
        rmse = np.sqrt(np.mean((predictions - actuals) ** 2))

        metrics = {
            'MAE': mae,
            'RMSE': rmse,
            'n_predictions': len(predictions)
        }

        print(f"MAE: {mae:.4f}")
        print(f"RMSE: {rmse:.4f}")
        print(f": {len(predictions)}")

        return metrics


class ItemKNNRecommender:

    def __init__(self, k: int = 20, similarity_metric: str = 'cosine'):
        self.k = k
        self.similarity_metric = similarity_metric
        self.item_similarity = None
        self.user_movie_matrix = None
        self.user_ids = None
        self.movie_ids = None

    def fit(self, user_movie_matrix: pd.DataFrame) -> None:
        print(f"\nTraining ItemKNN recommender (k={self.k}, metric={self.similarity_metric})...")
        start_time = time.time()

        self.user_movie_matrix = user_movie_matrix
        self.user_ids = user_movie_matrix.index
        self.movie_ids = user_movie_matrix.columns

        if self.similarity_metric == 'cosine':
            self.item_similarity = cosine_similarity(user_movie_matrix.T)
        elif self.similarity_metric == 'pearson':
            self.item_similarity = np.corrcoef(user_movie_matrix.T)
            self.item_similarity = np.nan_to_num(self.item_similarity, 0)
        else:
            raise ValueError(f": {self.similarity_metric}")

        self.item_similarity_df = pd.DataFrame(
            self.item_similarity,
            index=self.movie_ids,
            columns=self.movie_ids
        )

        print(f"Training complete！ {time.time() - start_time:.2f} ")
        print(f": {self.item_similarity.shape}")

    def predict_rating(self, user_id: int, movie_id: int) -> float:
        if user_id not in self.user_ids or movie_id not in self.movie_ids:
            return 2.5  # 

        user_ratings = self.user_movie_matrix.loc[user_id]
        rated_movies = user_ratings[user_ratings > 0]

        if len(rated_movies) == 0:
            return 2.5

        similarities = self.item_similarity_df.loc[movie_id, rated_movies.index]

        top_k_similar = similarities.nlargest(self.k)

        if len(top_k_similar) == 0 or top_k_similar.sum() == 0:
            return 2.5

        weighted_sum = 0
        similarity_sum = 0

        for similar_movie, similarity in top_k_similar.items():
            if similarity > 0:
                rating = rated_movies[similar_movie]
                weighted_sum += similarity * rating
                similarity_sum += similarity

        if similarity_sum > 0:
            prediction = weighted_sum / similarity_sum
        else:
            prediction = 2.5

        return np.clip(prediction, 0.5, 5.0)

    def recommend_for_user(self, user_id: int, top_n: int = 10,
                          exclude_rated: bool = True) -> List[Tuple[int, float]]:
        if user_id not in self.user_ids:
            raise ValueError(f" {user_id} ")

        user_ratings = self.user_movie_matrix.loc[user_id]
        rated_movies = user_ratings[user_ratings > 0]

        predictions = {}
        for movie_id in self.movie_ids:
            if exclude_rated and movie_id in rated_movies.index:
                continue

            pred_rating = self.predict_rating(user_id, movie_id)
            predictions[movie_id] = pred_rating

        top_movies = sorted(
            predictions.items(),
            key=lambda x: x[1],
            reverse=True
        )[:top_n]

        return top_movies

    def evaluate(self, test_ratings: pd.DataFrame) -> Dict[str, float]:
        print("\n ItemKNN ...")

        predictions = []
        actuals = []

        for _, row in test_ratings.iterrows():
            user_id = row['userId']
            movie_id = row['movieId']
            actual_rating = row['rating']

            try:
                pred_rating = self.predict_rating(user_id, movie_id)
                predictions.append(pred_rating)
                actuals.append(actual_rating)
            except (KeyError, ValueError):
                continue

        predictions = np.array(predictions)
        actuals = np.array(actuals)

        mae = np.mean(np.abs(predictions - actuals))
        rmse = np.sqrt(np.mean((predictions - actuals) ** 2))

        metrics = {
            'MAE': mae,
            'RMSE': rmse,
            'n_predictions': len(predictions)
        }

        print(f"MAE: {mae:.4f}")
        print(f"RMSE: {rmse:.4f}")
        print(f": {len(predictions)}")

        return metrics


## Module 4: Clustering

In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import dendrogram, linkage
from typing import Dict, List, Tuple, Optional
import time
from tqdm import tqdm


class MovieClusterer:

    def __init__(self):
        self.features = None
        self.movie_ids = None
        self.labels = None
        self.method = None
        self.n_clusters = None

    def kmeans_clustering(self, features: pd.DataFrame, n_clusters: int = 10,
                         random_state: int = 42) -> np.ndarray:
        print(f"\nExecuting K-means clustering (k={n_clusters})...")
        start_time = time.time()

        self.features = features
        self.movie_ids = features.index
        self.method = 'kmeans'
        self.n_clusters = n_clusters

        kmeans = KMeans(n_clusters=n_clusters, random_state=random_state,
                       n_init=10, max_iter=300)
        self.labels = kmeans.fit_predict(features)

        metrics = self.evaluate_clustering(features, self.labels)

        print(f"K-means clustering！ {time.time() - start_time:.2f} ")
        print(f": {metrics['silhouette']:.4f}")
        print(f"Davies-Bouldin : {metrics['davies_bouldin']:.4f}")
        print(f"Calinski-Harabasz : {metrics['calinski_harabasz']:.2f}")

        return self.labels

    def hierarchical_clustering(self, features: pd.DataFrame, n_clusters: int = 10,
                               linkage_method: str = 'ward') -> np.ndarray:
        print(f"\nExecutingclustering (n_clusters={n_clusters}, linkage={linkage_method})...")
        start_time = time.time()

        self.features = features
        self.movie_ids = features.index
        self.method = 'hierarchical'
        self.n_clusters = n_clusters

        hierarchical = AgglomerativeClustering(
            n_clusters=n_clusters,
            linkage=linkage_method
        )
        self.labels = hierarchical.fit_predict(features)

        metrics = self.evaluate_clustering(features, self.labels)

        print(f"clustering！ {time.time() - start_time:.2f} ")
        print(f": {metrics['silhouette']:.4f}")
        print(f"Davies-Bouldin : {metrics['davies_bouldin']:.4f}")
        print(f"Calinski-Harabasz : {metrics['calinski_harabasz']:.2f}")

        return self.labels

    @staticmethod
    def evaluate_clustering(features: np.ndarray, labels: np.ndarray) -> Dict[str, float]:
        silhouette = silhouette_score(features, labels)

        davies_bouldin = davies_bouldin_score(features, labels)

        calinski_harabasz = calinski_harabasz_score(features, labels)

        return {
            'silhouette': silhouette,
            'davies_bouldin': davies_bouldin,
            'calinski_harabasz': calinski_harabasz
        }

    def get_cluster_statistics(self, movie_info: pd.DataFrame) -> pd.DataFrame:
        if self.labels is None:
            raise ValueError("clustering")

        cluster_stats = []

        for cluster_id in range(self.n_clusters):
            cluster_mask = self.labels == cluster_id
            cluster_movie_ids = self.movie_ids[cluster_mask]

            cluster_movies = movie_info[movie_info['movieId'].isin(cluster_movie_ids)]

            stats = {
                'cluster_id': cluster_id,
                'size': len(cluster_movie_ids),
                'percentage': len(cluster_movie_ids) / len(self.labels) * 100,
            }

            if 'genres' in cluster_movies.columns:
                all_genres = []
                for genres in cluster_movies['genres']:
                    if genres != '(no genres listed)':
                        all_genres.extend(genres.split('|'))

                if all_genres:
                    from collections import Counter
                    genre_counts = Counter(all_genres)
                    top_genres = genre_counts.most_common(3)
                    stats['top_genres'] = ', '.join([f"{g}({c})" for g, c in top_genres])
                else:
                    stats['top_genres'] = 'N/A'

            cluster_stats.append(stats)

        return pd.DataFrame(cluster_stats)

    def find_optimal_k(self, features: pd.DataFrame, k_range: range = range(2, 21),
                      method: str = 'kmeans') -> Dict[int, Dict[str, float]]:
        print(f"\n k  (: {k_range.start}-{k_range.stop-1})...")

        results = {}

        for k in tqdm(k_range, desc="   k "):
            if method == 'kmeans':
                kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
                labels = kmeans.fit_predict(features)
                inertia = kmeans.inertia_
            else:
                hierarchical = AgglomerativeClustering(n_clusters=k)
                labels = hierarchical.fit_predict(features)
                inertia = None

            metrics = self.evaluate_clustering(features, labels)
            metrics['inertia'] = inertia

            results[k] = metrics

        return results


class DimensionalityReducer:

    def __init__(self, n_components: int = 50):
        self.n_components = n_components
        self.pca = None
        self.reduced_features = None
        self.original_features = None

    def fit_transform(self, features: pd.DataFrame) -> pd.DataFrame:
        n_features = features.shape[1]
        actual_n_components = min(self.n_components, n_features)

        if actual_n_components < self.n_components:
            print(f"\n:  ({n_features})  n_components ({self.n_components})")
            print(f" n_components={actual_n_components}")

        print(f"\n PCA  (n_components={actual_n_components})...")
        start_time = time.time()

        self.original_features = features
        self.pca = PCA(n_components=actual_n_components, random_state=42)
        reduced_data = self.pca.fit_transform(features)

        self.reduced_features = pd.DataFrame(
            reduced_data,
            index=features.index,
            columns=[f'PC{i+1}' for i in range(actual_n_components)]
        )

        explained_variance = np.sum(self.pca.explained_variance_ratio_)
        print(f"PCA ！ {time.time() - start_time:.2f} ")
        print(f": {features.shape[1]} -> : {actual_n_components}")
        print(f": {explained_variance:.4f}")

        return self.reduced_features

    def compare_clustering_with_without_pca(self, original_features: pd.DataFrame,
                                           n_clusters: int = 10,
                                           method: str = 'kmeans') -> Dict[str, Dict]:
        print(f"\nclustering...")

        reduced_features = self.fit_transform(original_features)

        clusterer_original = MovieClusterer()
        if method == 'kmeans':
            labels_original = clusterer_original.kmeans_clustering(original_features, n_clusters)
        else:
            labels_original = clusterer_original.hierarchical_clustering(original_features, n_clusters)

        metrics_original = MovieClusterer.evaluate_clustering(original_features, labels_original)

        clusterer_reduced = MovieClusterer()
        if method == 'kmeans':
            labels_reduced = clusterer_reduced.kmeans_clustering(reduced_features, n_clusters)
        else:
            labels_reduced = clusterer_reduced.hierarchical_clustering(reduced_features, n_clusters)

        metrics_reduced = MovieClusterer.evaluate_clustering(reduced_features, labels_reduced)

        print("\n:")
        print("=" * 60)
        print(f"{'':<25} {'':<15} {'PCA':<15}")
        print("=" * 60)
        print(f"{'':<25} {original_features.shape[1]:<15} {reduced_features.shape[1]:<15}")
        print(f"{' ()':<25} {metrics_original['silhouette']:<15.4f} {metrics_reduced['silhouette']:<15.4f}")
        print(f"{'Davies-Bouldin ()':<25} {metrics_original['davies_bouldin']:<15.4f} {metrics_reduced['davies_bouldin']:<15.4f}")
        print(f"{'Calinski-Harabasz ()':<25} {metrics_original['calinski_harabasz']:<15.2f} {metrics_reduced['calinski_harabasz']:<15.2f}")
        print("=" * 60)

        return {
            'original': {
                'metrics': metrics_original,
                'labels': labels_original,
                'features': original_features
            },
            'reduced': {
                'metrics': metrics_reduced,
                'labels': labels_reduced,
                'features': reduced_features
            },
            'pca': self.pca
        }


## Module 5: Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from typing import Dict, List
import os


plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

sns.set_style("whitegrid")
sns.set_palette("husl")


class Visualizer:

    def __init__(self, save_dir: str = './figures'):
        self.save_dir = save_dir
        os.makedirs(save_dir, exist_ok=True)

    def plot_elbow_curve(self, k_values: List[int], metrics: Dict[int, Dict],
                        metric_name: str = 'inertia', save_name: str = 'elbow_curve.png'):
        plt.figure(figsize=(10, 6))

        values = [metrics[k][metric_name] for k in k_values if metrics[k][metric_name] is not None]
        k_valid = [k for k in k_values if metrics[k][metric_name] is not None]

        plt.plot(k_valid, values, 'bo-', linewidth=2, markersize=8)
        plt.xlabel('Number of Clusters (k)', fontsize=12)
        plt.ylabel(metric_name.replace('_', ' ').title(), fontsize=12)
        plt.title(f'{metric_name.replace("_", " ").title()} vs Number of Clusters', fontsize=14)
        plt.grid(True, alpha=0.3)
        plt.tight_layout()

        save_path = os.path.join(self.save_dir, save_name)
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Figure saved: {save_path}")
        plt.close()

    def plot_clustering_comparison(self, metrics_dict: Dict[str, Dict],
                                   save_name: str = 'clustering_comparison.png'):
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))

        methods = list(metrics_dict.keys())
        metrics_names = ['silhouette', 'davies_bouldin', 'calinski_harabasz']
        titles = ['Silhouette Score\n(Higher is Better)',
                 'Davies-Bouldin Index\n(Lower is Better)',
                 'Calinski-Harabasz Index\n(Higher is Better)']

        for idx, (metric, title) in enumerate(zip(metrics_names, titles)):
            values = [metrics_dict[method][metric] for method in methods]

            axes[idx].bar(methods, values, alpha=0.7)
            axes[idx].set_title(title, fontsize=12)
            axes[idx].set_ylabel('Score', fontsize=10)
            axes[idx].tick_params(axis='x', rotation=45)
            axes[idx].grid(True, alpha=0.3, axis='y')

            for i, v in enumerate(values):
                axes[idx].text(i, v, f'{v:.3f}', ha='center', va='bottom', fontsize=9)

        plt.tight_layout()
        save_path = os.path.join(self.save_dir, save_name)
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Figure saved: {save_path}")
        plt.close()

    def plot_pca_variance(self, pca, save_name: str = 'pca_variance.png'):
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

        ax1.bar(range(1, len(pca.explained_variance_ratio_) + 1),
               pca.explained_variance_ratio_, alpha=0.7)
        ax1.set_xlabel('Principal Component', fontsize=12)
        ax1.set_ylabel('Explained Variance Ratio', fontsize=12)
        ax1.set_title('Variance Explained by Each Principal Component', fontsize=14)
        ax1.grid(True, alpha=0.3, axis='y')

        cumsum = np.cumsum(pca.explained_variance_ratio_)
        ax2.plot(range(1, len(cumsum) + 1), cumsum, 'ro-', linewidth=2, markersize=6)
        ax2.axhline(y=0.95, color='g', linestyle='--', label='95% Variance')
        ax2.axhline(y=0.90, color='b', linestyle='--', label='90% Variance')
        ax2.set_xlabel('Number of Components', fontsize=12)
        ax2.set_ylabel('Cumulative Explained Variance', fontsize=12)
        ax2.set_title('Cumulative Variance Explained', fontsize=14)
        ax2.legend()
        ax2.grid(True, alpha=0.3)

        plt.tight_layout()
        save_path = os.path.join(self.save_dir, save_name)
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Figure saved: {save_path}")
        plt.close()

    def plot_cluster_distribution(self, labels: np.ndarray,
                                  save_name: str = 'cluster_distribution.png'):
        plt.figure(figsize=(10, 6))

        unique, counts = np.unique(labels, return_counts=True)

        plt.bar(unique, counts, alpha=0.7)
        plt.xlabel('Cluster ID', fontsize=12)
        plt.ylabel('Number of Items', fontsize=12)
        plt.title('Distribution of Items Across Clusters', fontsize=14)
        plt.grid(True, alpha=0.3, axis='y')

        for i, (cluster_id, count) in enumerate(zip(unique, counts)):
            plt.text(cluster_id, count, str(count), ha='center', va='bottom', fontsize=9)

        plt.tight_layout()
        save_path = os.path.join(self.save_dir, save_name)
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Figure saved: {save_path}")
        plt.close()

    def plot_pca_2d_clusters(self, features: pd.DataFrame, labels: np.ndarray,
                            save_name: str = 'pca_2d_clusters.png'):
        from sklearn.decomposition import PCA

        pca_2d = PCA(n_components=2, random_state=42)
        features_2d = pca_2d.fit_transform(features)

        plt.figure(figsize=(10, 8))

        scatter = plt.scatter(features_2d[:, 0], features_2d[:, 1],
                            c=labels, cmap='tab10', alpha=0.6, s=30)
        plt.colorbar(scatter, label='Cluster')
        plt.xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.2%} variance)', fontsize=12)
        plt.ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.2%} variance)', fontsize=12)
        plt.title('Clusters Visualized in 2D PCA Space', fontsize=14)
        plt.grid(True, alpha=0.3)

        plt.tight_layout()
        save_path = os.path.join(self.save_dir, save_name)
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Figure saved: {save_path}")
        plt.close()

    def plot_recommendation_performance(self, metrics: Dict[str, float],
                                       save_name: str = 'recommendation_performance.png'):
        plt.figure(figsize=(8, 6))

        metric_names = list(metrics.keys())
        values = list(metrics.values())

        colors = ['#ff9999', '#66b3ff']
        bars = plt.bar(metric_names, values, alpha=0.7, color=colors)
        plt.ylabel('Error', fontsize=12)
        plt.title('Recommendation System Performance', fontsize=14)
        plt.grid(True, alpha=0.3, axis='y')

        for bar, val in zip(bars, values):
            height = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2., height,
                   f'{val:.4f}', ha='center', va='bottom', fontsize=11)

        plt.tight_layout()
        save_path = os.path.join(self.save_dir, save_name)
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Figure saved: {save_path}")
        plt.close()

    def plot_multi_metrics_comparison(self, results: Dict[str, Dict],
                                     save_name: str = 'pca_comparison.png'):
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))

        methods = ['Original Features', 'PCA Reduced']
        metrics_names = ['silhouette', 'davies_bouldin', 'calinski_harabasz']
        titles = ['Silhouette Score', 'Davies-Bouldin Index', 'Calinski-Harabasz Index']

        original_metrics = results['original']['metrics']
        reduced_metrics = results['reduced']['metrics']

        for idx, (metric, title) in enumerate(zip(metrics_names, titles)):
            values = [original_metrics[metric], reduced_metrics[metric]]

            bars = axes[idx].bar(methods, values, alpha=0.7, color=['#ff9999', '#66b3ff'])
            axes[idx].set_title(title, fontsize=12)
            axes[idx].set_ylabel('Score', fontsize=10)
            axes[idx].tick_params(axis='x', rotation=15)
            axes[idx].grid(True, alpha=0.3, axis='y')

            for bar, v in zip(bars, values):
                height = bar.get_height()
                axes[idx].text(bar.get_x() + bar.get_width()/2., height,
                             f'{v:.3f}', ha='center', va='bottom', fontsize=9)

        plt.tight_layout()
        save_path = os.path.join(self.save_dir, save_name)
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Figure saved: {save_path}")
        plt.close()


## Setup: Import Libraries

In [ ]:
import sys
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

print("Libraries imported successfully!")

## Step 1: Data Loading

In [ ]:
loader = MovieLensLoader(data_dir='./data')

try:
    loader.load_data(sample_size=2000000)
    print(f"Dataset loaded: {len(loader.ratings):,} ratings")
    print(f"Users: {loader.ratings['userId'].nunique():,}")
    print(f"Movies: {loader.ratings['movieId'].nunique():,}")
except FileNotFoundError:
    print("Error: Data files not found!")
    print("Download MovieLens 20M and extract to data/ml-20m/")

## Step 2: Data Preprocessing

In [ ]:
user_movie_matrix, filtered_ratings = loader.preprocess_for_recommendation(
    min_user_ratings=50,
    min_movie_ratings=50)

print(f"Matrix shape: {user_movie_matrix.shape}")
print(f"Total ratings: {len(filtered_ratings):,}")

sparsity = (1 - len(filtered_ratings) / (user_movie_matrix.shape[0] * user_movie_matrix.shape[1])) * 100
print(f"Sparsity: {sparsity:.2f}%")

## Step 3: Train-Test Split

In [ ]:
print("Splitting data...")

train_list = []
test_list = []

for user_id in filtered_ratings['userId'].unique():
    user_ratings = filtered_ratings[filtered_ratings['userId'] == user_id]
    n_test = max(1, int(len(user_ratings) * 0.2))
    user_ratings = user_ratings.sample(frac=1, random_state=42)
    test_list.append(user_ratings.iloc[:n_test])
    train_list.append(user_ratings.iloc[n_test:])

train_ratings = pd.concat(train_list, ignore_index=True)
test_ratings = pd.concat(test_list, ignore_index=True)

print(f"Train: {len(train_ratings):,} | Test: {len(test_ratings):,}")

train_matrix = train_ratings.pivot_table(
    index='userId',
    columns='movieId',
    values='rating',
    fill_value=0)

print(f"Train matrix: {train_matrix.shape}")

## Helper Function: Evaluate Ranking Quality

In [ ]:
def evaluate_ranking_quality(recommender, test_ratings, k=10, sample_users=500):
    user_relevant = create_relevance_set(test_ratings, threshold=4.0)
    sampled_users = list(user_relevant.keys())[:sample_users]

    user_recommendations = {}
    print(f"  Evaluating {len(sampled_users)} users...")
    for user_id in tqdm(sampled_users, desc="  Progress", leave=False):
        try:
            recs = recommender.recommend_for_user(user_id, top_n=k, exclude_rated=True)
            user_recommendations[user_id] = [movie_id for movie_id, _ in recs]
        except:
            continue

    evaluator = RecommenderEvaluator(k=k)
    metrics = evaluator.evaluate_recommendations(
        user_recommendations,
        {uid: user_relevant[uid] for uid in user_recommendations if uid in user_relevant},
        k=k
    )

    print(f"  Precision@{k}: {metrics[f'Precision@{k}']:.4f}")
    print(f"  Recall@{k}: {metrics[f'Recall@{k}']:.4f}")
    print(f"  NDCG@{k}: {metrics[f'NDCG@{k}']:.4f}")
    print(f"  HitRate@{k}: {metrics[f'HitRate@{k}']:.4f}")

    return metrics

print("Helper function defined!")

## Step 4: SVD Recommender

In [ ]:
print("="*80)
print("Training SVD Recommender")
print("="*80)

svd_recommender = SVDRecommender(n_components=200)
svd_recommender.fit(train_matrix)
svd_metrics = svd_recommender.evaluate(test_ratings)

print(f"MAE: {svd_metrics['MAE']:.4f} | RMSE: {svd_metrics['RMSE']:.4f}")

svd_ranking = evaluate_ranking_quality(svd_recommender, test_ratings, k=10, sample_users=500)

## Step 5: PageRank Recommender

In [ ]:
print("="*80)
print("Training PageRank Recommender")
print("="*80)

pagerank_recommender = PageRankRecommender(alpha=0.85, cf_weight=0.5)
pagerank_recommender.fit(train_matrix)
pr_metrics = pagerank_recommender.evaluate(test_ratings)

print(f"MAE: {pr_metrics['MAE']:.4f} | RMSE: {pr_metrics['RMSE']:.4f}")

pr_ranking = evaluate_ranking_quality(pagerank_recommender, test_ratings, k=10, sample_users=500)

## Step 6: Hybrid Recommender

In [ ]:
print("="*80)
print("Training Hybrid Recommender")
print("="*80)

hybrid_recommender = HybridRecommender(
    svd_recommender=svd_recommender,
    pagerank_recommender=pagerank_recommender,
    svd_weight=0.5)

hybrid_metrics = hybrid_recommender.evaluate(test_ratings)
print(f"MAE: {hybrid_metrics['MAE']:.4f} | RMSE: {hybrid_metrics['RMSE']:.4f}")

hybrid_ranking = evaluate_ranking_quality(hybrid_recommender, test_ratings, k=10, sample_users=500)

## Step 7: ALS Recommender

In [ ]:
print("="*80)
print("Training ALS Recommender")
print("="*80)

als_recommender = ALSRecommender(n_factors=100, n_iterations=10, regularization=0.01)
als_recommender.fit(train_matrix)
als_metrics = als_recommender.evaluate(test_ratings)

print(f"MAE: {als_metrics['MAE']:.4f} | RMSE: {als_metrics['RMSE']:.4f}")

als_ranking = evaluate_ranking_quality(als_recommender, test_ratings, k=10, sample_users=500)

## Step 8: ItemKNN Recommender

In [ ]:
print("="*80)
print("Training ItemKNN Recommender")
print("="*80)

itemknn_recommender = ItemKNNRecommender(k=30, similarity_metric='cosine')
itemknn_recommender.fit(train_matrix)
itemknn_metrics = itemknn_recommender.evaluate(test_ratings)

print(f"MAE: {itemknn_metrics['MAE']:.4f} | RMSE: {itemknn_metrics['RMSE']:.4f}")

itemknn_ranking = evaluate_ranking_quality(itemknn_recommender, test_ratings, k=10, sample_users=500)

## Performance Comparison

In [ ]:
print("="*100)
print("PERFORMANCE COMPARISON")
print("="*100)

print("\n[Rating Prediction] (Lower is Better)")
rating_comparison = pd.DataFrame({
    'Model': ['SVD (Traditional MF)', 'ALS (Modern MF)', 'ItemKNN (Traditional CF)', 'PageRank', 'Hybrid'],
    'MAE': [svd_metrics['MAE'], als_metrics['MAE'], itemknn_metrics['MAE'],
            pr_metrics['MAE'], hybrid_metrics['MAE']],
    'RMSE': [svd_metrics['RMSE'], als_metrics['RMSE'], itemknn_metrics['RMSE'],
             pr_metrics['RMSE'], hybrid_metrics['RMSE']]
})
print(rating_comparison.to_string(index=False, float_format='%.4f'))

print("\n[Ranking Quality] (Higher is Better)")
ranking_comparison = pd.DataFrame({
    'Model': ['SVD (Traditional MF)', 'ALS (Modern MF)', 'ItemKNN (Traditional CF)', 'PageRank', 'Hybrid'],
    'Precision@10': [svd_ranking['Precision@10'], als_ranking['Precision@10'],
                    itemknn_ranking['Precision@10'], pr_ranking['Precision@10'],
                    hybrid_ranking['Precision@10']],
    'Recall@10': [svd_ranking['Recall@10'], als_ranking['Recall@10'],
                 itemknn_ranking['Recall@10'], pr_ranking['Recall@10'],
                 hybrid_ranking['Recall@10']],
    'NDCG@10': [svd_ranking['NDCG@10'], als_ranking['NDCG@10'],
               itemknn_ranking['NDCG@10'], pr_ranking['NDCG@10'],
               hybrid_ranking['NDCG@10']],
    'HitRate@10': [svd_ranking['HitRate@10'], als_ranking['HitRate@10'],
                  itemknn_ranking['HitRate@10'], pr_ranking['HitRate@10'],
                  hybrid_ranking['HitRate@10']]
})
print(ranking_comparison.to_string(index=False, float_format='%.4f'))

## Visualization

In [ ]:
plt.figure(figsize=(14, 6))

models = ['SVD', 'ALS', 'ItemKNN', 'PageRank', 'Hybrid']
mae = [svd_metrics['MAE'], als_metrics['MAE'], itemknn_metrics['MAE'],
       pr_metrics['MAE'], hybrid_metrics['MAE']]
rmse = [svd_metrics['RMSE'], als_metrics['RMSE'], itemknn_metrics['RMSE'],
        pr_metrics['RMSE'], hybrid_metrics['RMSE']]

x = np.arange(len(models))
width = 0.35

plt.bar(x - width/2, mae, width, label='MAE', alpha=0.8, color='#2E86AB')
plt.bar(x + width/2, rmse, width, label='RMSE', alpha=0.8, color='#A23B72')

plt.xlabel('Recommendation System')
plt.ylabel('Error')
plt.title('Performance Comparison')
plt.xticks(x, models)
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()

plt.savefig('./figures/recommender_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## Sample Recommendations

In [ ]:
sample_user = train_matrix.index[10]
print(f"Recommendations for User {sample_user}:\n")

for name, rec in [('SVD', svd_recommender), ('PageRank', pagerank_recommender),
                  ('Hybrid', hybrid_recommender), ('ALS', als_recommender),
                  ('ItemKNN', itemknn_recommender)]:
    recs = rec.recommend_for_user(sample_user, top_n=5)
    print(f"{name}:")
    for i, (movie_id, score) in enumerate(recs, 1):
        info = loader.get_movie_info([movie_id])
        if len(info) > 0:
            print(f"  {i}. {info.iloc[0]['title']} ({score:.4f})")
    print()

## Clustering: K-means

In [ ]:
print("="*80)
print("CLUSTERING ANALYSIS")
print("="*80)

movie_features = loader.create_movie_features(use_genome=False)
print(f"Features: {movie_features.shape}")

clusterer_kmeans = MovieClusterer()
labels_kmeans = clusterer_kmeans.kmeans_clustering(movie_features, n_clusters=10)

## Clustering: Hierarchical

In [ ]:
clusterer_hierarchical = MovieClusterer()
labels_hierarchical = clusterer_hierarchical.hierarchical_clustering(movie_features, n_clusters=10)

## Clustering Comparison

In [ ]:
kmeans_metrics = MovieClusterer.evaluate_clustering(movie_features, labels_kmeans)
hierarchical_metrics = MovieClusterer.evaluate_clustering(movie_features, labels_hierarchical)

comparison_df = pd.DataFrame({
    'Method': ['K-means', 'Hierarchical'],
    'Silhouette': [kmeans_metrics['silhouette'], hierarchical_metrics['silhouette']],
    'Davies-Bouldin': [kmeans_metrics['davies_bouldin'], hierarchical_metrics['davies_bouldin']],
    'Calinski-Harabasz': [kmeans_metrics['calinski_harabasz'], hierarchical_metrics['calinski_harabasz']]
})
print(comparison_df.to_string(index=False))

viz = Visualizer(save_dir='./figures')
viz.plot_clustering_comparison({'K-means': kmeans_metrics, 'Hierarchical': hierarchical_metrics})

## Cluster Statistics

In [ ]:
stats = clusterer_kmeans.get_cluster_statistics(loader.movies)
print(stats.to_string())

viz.plot_cluster_distribution(labels_kmeans, 'kmeans_distribution.png')
viz.plot_pca_2d_clusters(movie_features, labels_kmeans, 'kmeans_2d.png')

## Optimal K Value

In [ ]:
optimal_k = clusterer_kmeans.find_optimal_k(movie_features, k_range=range(5, 21))

viz.plot_elbow_curve(list(optimal_k.keys()), optimal_k, 'silhouette', 'silhouette_elbow.png')
viz.plot_elbow_curve(list(optimal_k.keys()), optimal_k, 'davies_bouldin', 'davies_bouldin_elbow.png')

## PCA Dimensionality Reduction

In [ ]:
reducer = DimensionalityReducer(n_components=20)
comparison_results = reducer.compare_clustering_with_without_pca(
    movie_features, n_clusters=10, method='kmeans')

viz.plot_pca_variance(comparison_results['pca'])

## PCA Impact

In [ ]:
viz.plot_multi_metrics_comparison(comparison_results)

viz.plot_pca_2d_clusters(
    comparison_results['reduced']['features'],
    comparison_results['reduced']['labels'],
    'pca_reduced_2d.png')

## Summary

In [ ]:
print("="*80)
print("KEY FINDINGS")
print("="*80)

best = rating_comparison.loc[rating_comparison['MAE'].idxmin()]
print(f"\n1. Best Recommender: {best['Model']} (MAE={best['MAE']:.4f})")

print(f"\n2. Clustering: K-means Silhouette={kmeans_metrics['silhouette']:.4f}, Hierarchical={hierarchical_metrics['silhouette']:.4f}")

orig_sil = comparison_results['original']['metrics']['silhouette']
pca_sil = comparison_results['reduced']['metrics']['silhouette']
print(f"\n3. PCA Impact: Original={orig_sil:.4f}, PCA={pca_sil:.4f}")
print(f"   {'PCA improves' if pca_sil > orig_sil else 'Original better'}")

print("\n4. All visualizations saved to ./figures/")
print("="*80)